# Module 05 — Lecture 2: Sparse Connectivity on GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_05_networks_plasticity/02_sparse_connectivity.ipynb)

---

Cortical networks are **sparse**: each neuron contacts ~10% of local neurons. Storing a full N×N weight matrix wastes memory and compute. This lecture covers:

1. **CSR format** — the standard for sparse matrices on GPU
2. **Sparse matrix-vector multiply (SpMV)** — the core kernel for computing synaptic input
3. **Performance comparison** — dense vs sparse for realistic network sizes

In [ ]:
!nvidia-smi

## 1. Why Sparse?

For N=10,000 neurons with 10% connectivity:

| Representation | Memory | Cost per step |
|----------------|--------|---------------|
| Dense N×N matrix | 400 MB (float32) | N² multiply-adds |
| Sparse (CSR, p=0.1) | ~40 MB | N × 0.1N = 0.1N² |

For N=100,000 (large-scale simulation):
- Dense: **40 GB** — doesn't fit on GPU
- Sparse: **400 MB** — fits comfortably

## 2. CSR Format

**Compressed Sparse Row (CSR)** stores only non-zero entries:

```
Matrix W (4×4, p=0.5):
  0  0.3  0   0.1
  0   0  0.2   0
 0.4  0   0   0.5
  0  0.1  0    0

CSR:
  row_ptr = [0, 2, 3, 5, 6]   # start index of each row's entries
  col_idx = [1, 3, 2, 0, 3, 1]  # column of each non-zero
  values  = [0.3, 0.1, 0.2, 0.4, 0.5, 0.1]  # non-zero weights
```

**Key property:** row_ptr has N+1 entries. The synapses from neuron i run from `row_ptr[i]` to `row_ptr[i+1]-1`.

**SpMV** (computing synaptic input to each neuron):
$$I_{syn}[i] = \sum_{j: W_{ij} \neq 0} W_{ij} \cdot s[j]$$

where $s[j]=1$ if neuron $j$ fired this timestep.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Build a random sparse connectivity matrix
np.random.seed(42)
N = 500
p = 0.1

# Random sparse adjacency
W_dense = (np.random.rand(N, N) < p).astype(np.float32)
np.fill_diagonal(W_dense, 0)
W_dense *= np.random.exponential(0.1, (N, N)).astype(np.float32)

# Convert to CSR
W_csr = sp.csr_matrix(W_dense)
print(f"Dense matrix: {N*N*4/1e6:.1f} MB")
print(f"CSR matrix:   {(W_csr.data.nbytes + W_csr.indices.nbytes + W_csr.indptr.nbytes)/1e6:.2f} MB")
print(f"Compression:  {N*N / len(W_csr.data):.1f}x")
print(f"Synapses:     {W_csr.nnz} ({100*W_csr.nnz/(N*N):.1f}% density)")

# Visualize the sparsity pattern
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sample = 100
axes[0].spy(W_dense[:sample, :sample], markersize=2)
axes[0].set_title(f'Connectivity (first {sample}×{sample}, p={p})', fontsize=12)
axes[0].set_xlabel('Pre-synaptic neuron')
axes[0].set_ylabel('Post-synaptic neuron')

# Degree distribution
in_degrees  = (W_dense > 0).sum(axis=0)
out_degrees = (W_dense > 0).sum(axis=1)
axes[1].hist(out_degrees, bins=30, alpha=0.7, label='Out-degree', color='steelblue')
axes[1].hist(in_degrees,  bins=30, alpha=0.7, label='In-degree',  color='coral')
axes[1].axvline(N*p, color='k', linestyle='--', label=f'Mean = {N*p:.0f}')
axes[1].set_xlabel('Degree', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Degree Distribution', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sparse_connectivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Building CSR on CPU, Transferring to GPU

The standard pipeline:
1. Build connectivity on CPU (one-time setup)
2. Transfer `row_ptr`, `col_idx`, `weights` to GPU device arrays
3. Use them in every simulation timestep

The `network_sim.cu` uses a 2-pass construction:
- **Pass 1**: count connections per row → compute `row_ptr`
- **Pass 2**: fill `col_idx` and `weights` using same RNG seed

In [ ]:
%%writefile csr_spmv.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e)); \
        exit(1); } } while(0)

// ─────────────────────────────────────────────────────────────────────────────
// Dense SpMV: I[i] = sum_j W[i*N + j] * s[j]
// Thread i computes one output element — classic row-parallel pattern
// ─────────────────────────────────────────────────────────────────────────────
__global__ void spmv_dense(
    const float* __restrict__ W, // [N*N]
    const float* __restrict__ s, // [N] spike vector
    float* __restrict__ I,       // [N] output
    int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float sum = 0.0f;
    for (int j = 0; j < N; j++) sum += W[i * N + j] * s[j];
    I[i] = sum;
}

// ─────────────────────────────────────────────────────────────────────────────
// Sparse SpMV (CSR): I[i] = sum over row i's non-zeros
// Thread i processes row i — works well when rows are roughly equal length
// ─────────────────────────────────────────────────────────────────────────────
__global__ void spmv_csr(
    const int*   __restrict__ row_ptr, // [N+1]
    const int*   __restrict__ col_idx, // [nnz]
    const float* __restrict__ values,  // [nnz]
    const float* __restrict__ s,       // [N]
    float* __restrict__ I,             // [N]
    int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float sum = 0.0f;
    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++) {
        sum += values[k] * s[col_idx[k]];
    }
    I[i] = sum;
}

// ─────────────────────────────────────────────────────────────────────────────
// Build random CSR connectivity on CPU
// ─────────────────────────────────────────────────────────────────────────────
void build_csr(int N, float p,
               int** row_ptr, int** col_idx, float** vals, int* nnz_out)
{
    srand(42);
    int* cnt = (int*)calloc(N, sizeof(int));
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            if (i != j && (float)rand()/RAND_MAX < p) cnt[i]++;

    *row_ptr = (int*)malloc((N+1)*sizeof(int));
    (*row_ptr)[0] = 0;
    for (int i = 0; i < N; i++) (*row_ptr)[i+1] = (*row_ptr)[i] + cnt[i];
    int nnz = (*row_ptr)[N];
    *nnz_out = nnz;

    *col_idx = (int*)malloc(nnz*sizeof(int));
    *vals    = (float*)malloc(nnz*sizeof(float));

    srand(42);
    int* pos = (int*)calloc(N, sizeof(int));
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            if (i != j && (float)rand()/RAND_MAX < p) {
                int k = (*row_ptr)[i] + pos[i]++;
                (*col_idx)[k] = j;
                (*vals)[k]    = 0.1f;  // uniform weight
            }
        }
    }
    free(cnt); free(pos);
}

int main(int argc, char** argv)
{
    int N = (argc > 1) ? atoi(argv[1]) : 2000;
    float p = 0.1f;
    int REPS = 100;

    // Build CSR
    int *h_row_ptr, *h_col_idx, nnz;
    float* h_vals;
    build_csr(N, p, &h_row_ptr, &h_col_idx, &h_vals, &nnz);

    // Random spike vector (10% firing)
    float* h_s = (float*)calloc(N, sizeof(float));
    srand(123);
    for (int i = 0; i < N; i++) if ((float)rand()/RAND_MAX < 0.05f) h_s[i] = 1.0f;

    printf("N=%d, nnz=%d (%.1f per neuron)\n", N, nnz, (float)nnz/N);

    // Dense matrix for comparison
    float* h_W = (float*)calloc((size_t)N*N, sizeof(float));
    for (int i = 0; i < N; i++)
        for (int k = h_row_ptr[i]; k < h_row_ptr[i+1]; k++)
            h_W[(size_t)i*N + h_col_idx[k]] = h_vals[k];

    // Allocate GPU
    float *d_W, *d_s, *d_I;
    int   *d_row_ptr, *d_col_idx;
    float *d_vals;

    CUDA_CHECK(cudaMalloc(&d_s,       N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_I,       N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_W,  (size_t)N*N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_row_ptr, (N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_col_idx, nnz*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_vals,    nnz*sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_s,       h_s,       N*sizeof(float),        cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_W,       h_W,  (size_t)N*N*sizeof(float),   cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_row_ptr, h_row_ptr, (N+1)*sizeof(int),      cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_col_idx, h_col_idx, nnz*sizeof(int),        cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_vals,    h_vals,    nnz*sizeof(float),      cudaMemcpyHostToDevice));

    int thr = 256, blk = (N + thr - 1) / thr;
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    float ms;

    // Benchmark dense
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r = 0; r < REPS; r++) spmv_dense<<<blk, thr>>>(d_W, d_s, d_I, N);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));
    float dense_ms = ms / REPS;
    float dense_bw = (float)N*N*2*sizeof(float) / (dense_ms * 1e6f);  // GB/s
    printf("Dense SpMV:  %.3f ms per call, %.1f GB/s effective BW\n", dense_ms, dense_bw);

    // Benchmark sparse CSR
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r = 0; r < REPS; r++) spmv_csr<<<blk, thr>>>(d_row_ptr, d_col_idx, d_vals, d_s, d_I, N);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));
    float sparse_ms = ms / REPS;
    float sparse_bw = (float)nnz*2*sizeof(float) / (sparse_ms * 1e6f);
    printf("Sparse SpMV: %.3f ms per call, %.1f GB/s effective BW\n", sparse_ms, sparse_bw);
    printf("Speedup:     %.1fx\n", dense_ms / sparse_ms);

    // Memory usage
    printf("Dense memory: %.1f MB\n", (float)N*N*4/1e6f);
    printf("Sparse memory: %.2f MB\n", (float)(nnz*8 + (N+1)*4)/1e6f);

    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_W); cudaFree(d_s); cudaFree(d_I);
    cudaFree(d_row_ptr); cudaFree(d_col_idx); cudaFree(d_vals);
    free(h_W); free(h_s); free(h_row_ptr); free(h_col_idx); free(h_vals);
    return 0;
}

In [ ]:
!nvcc -O2 -o csr_spmv csr_spmv.cu -lm && ./csr_spmv 2000

## 4. Performance vs Network Size

The key question: when does sparse beat dense, and by how much?

**Analysis:**
- Dense SpMV reads N² floats → bandwidth limited at N² × 4 bytes
- Sparse SpMV reads ≈ p × N² floats (p=0.1 → 10% of dense work)
- But sparse has **irregular memory access** to `s[col_idx[k]]` — random reads hurt L2 hit rate

For random connectivity with p=0.1:
- Small N (≤512): dense may be comparable (overhead dominates)
- Large N (≥2000): sparse wins clearly on both time and memory

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

# Sweep over network sizes
# Note: dense N=5000 needs 100MB — may be slow on small GPUs
Ns = [500, 1000, 2000, 3000]

dense_times  = []
sparse_times = []

for N in Ns:
    result = subprocess.run(['./csr_spmv', str(N)],
                            capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    for line in lines:
        if 'Dense' in line:
            dense_times.append(float(line.split()[2]))
        elif 'Sparse' in line and 'SpMV' in line:
            sparse_times.append(float(line.split()[2]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(Ns, dense_times,  'b-o', lw=2, label='Dense SpMV')
ax1.loglog(Ns, sparse_times, 'r-s', lw=2, label='CSR SpMV')
ax1.set_xlabel('Network size N', fontsize=12)
ax1.set_ylabel('Time per call (ms)', fontsize=12)
ax1.set_title('SpMV Kernel Time vs N', fontsize=13)
ax1.legend(); ax1.grid(True, which='both', alpha=0.3)

speedups = [d/s for d, s in zip(dense_times, sparse_times)]
ax2.plot(Ns, speedups, 'g-^', lw=2, markersize=8)
ax2.axhline(1, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('Network size N', fontsize=12)
ax2.set_ylabel('Dense / Sparse time', fontsize=12)
ax2.set_title('Sparse Speedup (p=0.1)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spmv_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Memory savings
print("\nMemory comparison (p=0.1):")
for N in Ns:
    dense_mb  = N*N*4 / 1e6
    sparse_mb = int(N*N*0.1) * 8 / 1e6  # col_idx (4B) + values (4B)
    print(f"  N={N:5d}: dense={dense_mb:7.1f} MB, sparse={sparse_mb:5.1f} MB "
          f"({dense_mb/sparse_mb:.1f}x smaller)")

## 5. Spike Propagation vs SpMV

In a spiking network, only ~1-5% of neurons fire per timestep. We have two equivalent approaches:

**Approach A — SpMV (pull):** Thread i sums incoming weights × s[j]
```
I[i] = sum_j W[i,j] * s[j]    // every neuron reads its full row
```

**Approach B — Scatter (push):** For each fired neuron j, scatter to its targets
```
for each j with s[j]=1:        // only 2% of neurons do work
    for k in row_ptr_out[j]:
        atomicAdd(&g[col_idx[k]], weights[k])
```

When firing rate < ~50%, the scatter approach is faster because:
- Most threads have nothing to do → skip immediately
- Reads only the outgoing edges of fired neurons
- This is what `propagate_spikes` in `network_sim.cu` does

### Thread divergence caveat

In `propagate_spikes`, the loop `for k in row_ptr_out[i]...` has different lengths per thread. This creates **warp divergence** when some threads have many outgoing connections and others have few. For very unequal degree distributions, warp-level load balancing (e.g., one warp per row) is needed.

In [ ]:
%%writefile scatter_vs_spmv.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

// SpMV pull: every neuron reads its incoming row
__global__ void spmv_pull(
    const int* row_ptr, const int* col_idx, const float* vals,
    const float* s, float* I, int N)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float acc = 0.f;
    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++)
        acc += vals[k] * s[col_idx[k]];
    I[i] = acc;
}

// Scatter push: only fired neurons scatter their output weights
__global__ void scatter_push(
    const int* fired_list, int n_fired,
    const int* row_ptr, const int* col_idx, const float* vals,
    float* g, int N)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n_fired) return;
    int i = fired_list[tid];
    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++)
        atomicAdd(&g[col_idx[k]], vals[k]);
}

int main(int argc, char** argv)
{
    int N = (argc>1) ? atoi(argv[1]) : 2000;
    float p = 0.1f, fire_rate = 0.02f;

    // Build CSR (outgoing from each neuron)
    srand(42);
    int* cnt = (int*)calloc(N, sizeof(int));
    for (int i=0;i<N;i++)
        for (int j=0;j<N;j++)
            if (i!=j && (float)rand()/RAND_MAX < p) cnt[i]++;

    int* rp = (int*)malloc((N+1)*sizeof(int)); rp[0]=0;
    for (int i=0;i<N;i++) rp[i+1]=rp[i]+cnt[i];
    int nnz = rp[N];
    int* ci = (int*)malloc(nnz*sizeof(int));
    float* wv = (float*)malloc(nnz*sizeof(float));

    srand(42);
    int* pos=(int*)calloc(N,sizeof(int));
    for (int i=0;i<N;i++) for (int j=0;j<N;j++)
        if (i!=j && (float)rand()/RAND_MAX < p) {
            int k=rp[i]+pos[i]++; ci[k]=j; wv[k]=0.1f; }
    free(cnt); free(pos);

    // Spike vector + fired list
    float* h_s = (float*)calloc(N, sizeof(float));
    int*   h_fl= (int*)malloc(N*sizeof(int));
    int n_fired = 0;
    srand(99);
    for (int i=0;i<N;i++) if ((float)rand()/RAND_MAX < fire_rate) {
        h_s[i]=1.f; h_fl[n_fired++]=i; }
    printf("Fired: %d / %d = %.1f%%\n", n_fired, N, 100.f*n_fired/N);

    // GPU alloc
    int *d_rp,*d_ci,*d_fl; float *d_wv,*d_s,*d_I,*d_g;
    CUDA_CHECK(cudaMalloc(&d_rp,(N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_ci,nnz*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_wv,nnz*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_s, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_I, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_g, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_fl,N*sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_rp,rp,(N+1)*sizeof(int),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_ci,ci,nnz*sizeof(int),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_wv,wv,nnz*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_s, h_s,N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_fl,h_fl,n_fired*sizeof(int),cudaMemcpyHostToDevice));

    int thr=256, REPS=200;
    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));

    // SpMV pull
    int blk_N = (N+thr-1)/thr;
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r=0;r<REPS;r++) spmv_pull<<<blk_N,thr>>>(d_rp,d_ci,d_wv,d_s,d_I,N);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    printf("SpMV pull:    %.3f ms/call\n", ms/REPS);

    // Scatter push
    int blk_f = (n_fired+thr-1)/thr;
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r=0;r<REPS;r++) {
        CUDA_CHECK(cudaMemset(d_g,0,N*sizeof(float)));
        scatter_push<<<blk_f,thr>>>(d_fl,n_fired,d_rp,d_ci,d_wv,d_g,N);
    }
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    printf("Scatter push: %.3f ms/call (includes memset)\n", ms/REPS);

    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_rp);cudaFree(d_ci);cudaFree(d_wv);
    cudaFree(d_s);cudaFree(d_I);cudaFree(d_g);cudaFree(d_fl);
    free(rp);free(ci);free(wv);free(h_s);free(h_fl);
    return 0;
}

In [ ]:
!nvcc -O2 -o scatter_vs_spmv scatter_vs_spmv.cu && ./scatter_vs_spmv 2000

## 6. Constructing Structured Connectivity

Random connectivity is the simplest case. Real networks have structure:
- **Distance-dependent** probability (nearby neurons more likely connected)
- **Cell-type specific** (PV interneurons target soma, SST targets dendrites)
- **Columnar** organization (within-column dense, across-column sparse)

The CSR format accommodates all of these — construction just changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N = 200

# Place neurons on a 1D ring [0, 2π)
theta = np.linspace(0, 2*np.pi, N, endpoint=False)

# Distance-dependent connectivity: p(d) = p0 * exp(-d^2 / 2sigma^2)
p0 = 0.5
sigma = 0.3  # in radians

rows, cols = [], []
for i in range(N):
    for j in range(N):
        if i == j: continue
        d = min(abs(theta[i] - theta[j]), 2*np.pi - abs(theta[i] - theta[j]))
        p = p0 * np.exp(-d**2 / (2*sigma**2))
        if np.random.rand() < p:
            rows.append(i); cols.append(j)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Connectivity matrix (reordered by theta)
W_ring = np.zeros((N, N))
for r, c in zip(rows, cols): W_ring[r, c] = 1

axes[0].imshow(W_ring, cmap='Blues', aspect='auto')
axes[0].set_title('Ring Network Connectivity\n(distance-dependent)', fontsize=12)
axes[0].set_xlabel('Pre-synaptic neuron'); axes[0].set_ylabel('Post-synaptic neuron')

# Connection probability vs distance
d_vals = np.linspace(0, np.pi, 100)
p_vals = p0 * np.exp(-d_vals**2 / (2*sigma**2))
axes[1].plot(np.degrees(d_vals), p_vals, 'b-', lw=2)
axes[1].set_xlabel('Angular distance (degrees)', fontsize=12)
axes[1].set_ylabel('Connection probability', fontsize=12)
axes[1].set_title('Distance-Dependent Connectivity Profile', fontsize=12)
axes[1].grid(True, alpha=0.3)

print(f"Ring network: {len(rows)} synapses, density = {len(rows)/(N*N)*100:.1f}%")
plt.tight_layout()
plt.savefig('ring_connectivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Format | Memory | Access pattern | When to use |
|--------|--------|----------------|-------------|
| Dense (N×N) | O(N²) | Coalesced rows | p > 0.3, small N |
| CSR (SpMV pull) | O(nnz) | Random col access | General sparse |
| CSR (scatter push) | O(nnz) | Output-sparse | Low firing rates |

**Key insight:** In a spiking network with 2% firing rate, scatter push touches only 2% of rows each timestep — a 50× work reduction over SpMV pull.

**Next lecture:** Spike-Timing-Dependent Plasticity (STDP) — making the weights in those CSR arrays change over time.